In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_test = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/test.csv")
df_test.head()

,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


In [3]:
df_test.shape

(28512, 5)

In [4]:
df_test.columns

Index(['id', 'date', 'store_nbr', 'family', 'onpromotion'], dtype='object')

In [5]:
df_test['date'].unique()

array(['2017-08-16', '2017-08-17', '2017-08-18', '2017-08-19',
       '2017-08-20', '2017-08-21', '2017-08-22', '2017-08-23',
       '2017-08-24', '2017-08-25', '2017-08-26', '2017-08-27',
       '2017-08-28', '2017-08-29', '2017-08-30', '2017-08-31'],
      dtype=object)

Create Calendar Feature

In [6]:
df_test["date"] = pd.to_datetime(df_test["date"])

df_test["day_of_week"] = df_test["date"].dt.dayofweek
df_test["month"] = df_test["date"].dt.month
df_test["year"] = df_test["date"].dt.year
df_test["week_of_year"] = df_test["date"].dt.isocalendar().week
df_test["is_weekend"] = df_test["day_of_week"].isin([5,6]).astype(int)
df_test["day_of_month"] = df_test["date"].dt.day
df_test["is_month_start"] = df_test["date"].dt.is_month_start.astype(int)
df_test["is_month_end"] = df_test["date"].dt.is_month_end.astype(int)

In [7]:
df_test.head()

,id,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,day_of_month,is_month_start,is_month_end
0,3000888,2017-08-16,1,AUTOMOTIVE,0,2,8,2017,33,0,16,0,0
1,3000889,2017-08-16,1,BABY CARE,0,2,8,2017,33,0,16,0,0
2,3000890,2017-08-16,1,BEAUTY,2,2,8,2017,33,0,16,0,0
3,3000891,2017-08-16,1,BEVERAGES,20,2,8,2017,33,0,16,0,0
4,3000892,2017-08-16,1,BOOKS,0,2,8,2017,33,0,16,0,0


Add holiday feature

In [8]:
df_holiday = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/holidays_events.csv")
df_holiday.head(20)

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


In [9]:
df_holidays_clean = df_holiday[
    (df_holiday["transferred"] == False) &
    (df_holiday["type"] != "Work Day")
]
print(f"before clean :{df_holiday.shape}")
print(f"after clean :{df_holidays_clean.shape}")

before clean :(350, 6)
after clean :(333, 6)


In [10]:
holiday_types = ["Holiday", "Transfer", "Bridge", "Additional"]

holiday_flags = (
    df_holidays_clean
    .assign(
        is_holiday=lambda x: x["type"].isin(holiday_types),
        is_earthquake=lambda x: x["description"].str.contains("Terremoto", na=False),
        is_other_event=lambda x: (x["type"]=="Event") & (~x["description"].str.contains("Terremoto", na=False))
    )
    .groupby("date")[["is_holiday","is_earthquake","is_other_event"]]
    .max()
    .reset_index()
)

holiday_flags.head()

,date,is_holiday,is_earthquake,is_other_event
0,2012-03-02,True,False,False
1,2012-04-01,True,False,False
2,2012-04-12,True,False,False
3,2012-04-14,True,False,False
4,2012-04-21,True,False,False


In [11]:
print(f"Before merge :{df_test.shape}")
holiday_flags["date"] = pd.to_datetime(holiday_flags["date"])
df_test = df_test.merge(holiday_flags, on="date", how="left")
print(f"After merge :{df_test.shape}")

Before merge :(28512, 13)
After merge :(28512, 16)


In [12]:
df_test[['is_holiday',	'is_earthquake',	'is_other_event']].isnull().sum()

is_holiday        26730
is_earthquake     26730
is_other_event    26730
dtype: int64

In [13]:
df_test[['is_holiday',	'is_earthquake',	'is_other_event']].notnull().sum()

is_holiday        1782
is_earthquake     1782
is_other_event    1782
dtype: int64

Fill NaN in 'is_holiday',	'is_earthquake',	'is_other_event' columns with 0

In [14]:
df_test[["is_holiday", "is_earthquake", "is_other_event"]] = (
    df_test[["is_holiday", "is_earthquake", "is_other_event"]]
    .fillna(0)
    .astype(int)
)

In [15]:
print(f"After fill Nan : {df_test[['is_holiday',	'is_earthquake',	'is_other_event']].isnull().sum()}")

After fill Nan : is_holiday        0
is_earthquake     0
is_other_event    0
dtype: int64


In [16]:
df_test[["is_holiday","is_earthquake","is_other_event"]].sum()

is_holiday        1782
is_earthquake        0
is_other_event       0
dtype: int64

Add promotion feature flag

In [17]:
df_test["is_promo"] = (df_test["onpromotion"] > 0).astype(int)
df_test.head()

,id,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,day_of_month,is_month_start,is_month_end,is_holiday,is_earthquake,is_other_event,is_promo
0,3000888,2017-08-16,1,AUTOMOTIVE,0,2,8,2017,33,0,16,0,0,0,0,0,0
1,3000889,2017-08-16,1,BABY CARE,0,2,8,2017,33,0,16,0,0,0,0,0,0
2,3000890,2017-08-16,1,BEAUTY,2,2,8,2017,33,0,16,0,0,0,0,0,1
3,3000891,2017-08-16,1,BEVERAGES,20,2,8,2017,33,0,16,0,0,0,0,0,1
4,3000892,2017-08-16,1,BOOKS,0,2,8,2017,33,0,16,0,0,0,0,0,0


Add feature lag feature

In [18]:
df_train = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/train.csv")
df_train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [19]:
df_train.tail()

,id,date,store_nbr,family,sales,onpromotion
3000883,3000883,2017-08-15,9,POULTRY,438.133,0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8
3000887,3000887,2017-08-15,9,SEAFOOD,16.000,0


In [20]:
df_train['date'] = pd.to_datetime(df_train['date'])

df_train["is_test"] = 0
df_test["is_test"] = 1
df_test["sales"] = np.nan  # placeholder

combined = pd.concat([df_train, df_test]).sort_values(["store_nbr", "family", "date"])

for lag in [7, 14, 28]:
    combined[f"sales_lag_{lag}"] = (
        combined.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

for window in [7, 14, 28]:
    combined[f"sales_rolling_mean_{window}"] = (
        combined.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

combined["sales_rolling_std_7"] = (
    combined.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

combined["promo_lag_7"] = (
    combined.groupby(["store_nbr", "family"])["onpromotion"]
    .shift(7)
)

# Split back
df_test = combined[combined["is_test"] == 1].drop(columns=["sales", "is_test"])
df_test.head()


,id,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,...,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7
0,3000888,2017-08-16,1,AUTOMOTIVE,0,2.0,8.0,2017.0,33,0.0,...,0.0,0.0,7.0,4.0,7.0,4.142857,4.714286,5.035714,3.287784,0.0
1782,3002670,2017-08-17,1,AUTOMOTIVE,0,3.0,8.0,2017.0,33,0.0,...,0.0,0.0,9.0,3.0,4.0,NaN,NaN,NaN,NaN,0.0
3564,3004452,2017-08-18,1,AUTOMOTIVE,0,4.0,8.0,2017.0,33,0.0,...,0.0,0.0,1.0,8.0,10.0,NaN,NaN,NaN,NaN,0.0
5346,3006234,2017-08-19,1,AUTOMOTIVE,0,5.0,8.0,2017.0,33,1.0,...,0.0,0.0,6.0,5.0,8.0,NaN,NaN,NaN,NaN,0.0
7128,3008016,2017-08-20,1,AUTOMOTIVE,0,6.0,8.0,2017.0,33,1.0,...,0.0,0.0,1.0,6.0,0.0,NaN,NaN,NaN,NaN,0.0


In [21]:
df_test.tail()

,id,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,...,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7
21251,3022139,2017-08-27,54,SEAFOOD,0,6.0,8.0,2017.0,34,1.0,...,0.0,0.0,NaN,2.0,4.0,NaN,NaN,NaN,NaN,0.0
23033,3023921,2017-08-28,54,SEAFOOD,0,0.0,8.0,2017.0,35,0.0,...,0.0,0.0,NaN,0.0,4.0,NaN,NaN,NaN,NaN,0.0
24815,3025703,2017-08-29,54,SEAFOOD,0,1.0,8.0,2017.0,35,0.0,...,0.0,0.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,0.0
26597,3027485,2017-08-30,54,SEAFOOD,0,2.0,8.0,2017.0,35,0.0,...,0.0,0.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,0.0
28379,3029267,2017-08-31,54,SEAFOOD,0,3.0,8.0,2017.0,35,0.0,...,0.0,0.0,NaN,NaN,5.0,NaN,NaN,NaN,NaN,0.0


In [22]:
df_test[["sales_lag_7", "sales_lag_14", "sales_lag_28","sales_rolling_mean_7",	"sales_rolling_mean_14",	"sales_rolling_mean_28",	"sales_rolling_std_7"]].isnull().sum()

sales_lag_7              16038
sales_lag_14              3564
sales_lag_28                 0
sales_rolling_mean_7     26730
sales_rolling_mean_14    26730
sales_rolling_mean_28    26730
sales_rolling_std_7      26730
dtype: int64

Add store feature

In [23]:
df_stores = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/stores.csv")
df_test = df_test.merge(df_stores, on="store_nbr", how="left")
df_test.head()

,id,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,...,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7,city,state,type,cluster
0,3000888,2017-08-16,1,AUTOMOTIVE,0,2.0,8.0,2017.0,33,0.0,...,7.0,4.142857,4.714286,5.035714,3.287784,0.0,Quito,Pichincha,D,13
1,3002670,2017-08-17,1,AUTOMOTIVE,0,3.0,8.0,2017.0,33,0.0,...,4.0,NaN,NaN,NaN,NaN,0.0,Quito,Pichincha,D,13
2,3004452,2017-08-18,1,AUTOMOTIVE,0,4.0,8.0,2017.0,33,0.0,...,10.0,NaN,NaN,NaN,NaN,0.0,Quito,Pichincha,D,13
3,3006234,2017-08-19,1,AUTOMOTIVE,0,5.0,8.0,2017.0,33,1.0,...,8.0,NaN,NaN,NaN,NaN,0.0,Quito,Pichincha,D,13
4,3008016,2017-08-20,1,AUTOMOTIVE,0,6.0,8.0,2017.0,33,1.0,...,0.0,NaN,NaN,NaN,NaN,0.0,Quito,Pichincha,D,13


Add oil feature

In [24]:
df_oil = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/oil.csv")
df_oil["date"] = pd.to_datetime(df_oil["date"])
df_test = df_test.merge(df_oil, on="date", how="left")

In [25]:
print("The number of NaN in oil column :", df_test["dcoilwtico"].isnull().sum()) 

The number of NaN in oil column : 7128


In [26]:
df_test["dcoilwtico"] = df_test["dcoilwtico"].ffill().bfill()
print("The number of NaN in oil column :", df_test["dcoilwtico"].isnull().sum()) 

The number of NaN in oil column : 0


In [27]:
feature_cols = [col for col in df_test.columns if col not in ["id", "date", "sales"]]

X_test = df_test[feature_cols]

In [28]:
len(X_test.columns)

28

In [29]:
X_test.columns

Index(['store_nbr', 'family', 'onpromotion', 'day_of_week', 'month', 'year',
       'week_of_year', 'is_weekend', 'day_of_month', 'is_month_start',
       'is_month_end', 'is_holiday', 'is_earthquake', 'is_other_event',
       'is_promo', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
       'sales_rolling_mean_7', 'sales_rolling_mean_14',
       'sales_rolling_mean_28', 'sales_rolling_std_7', 'promo_lag_7', 'city',
       'state', 'type', 'cluster', 'dcoilwtico'],
      dtype='object')

In [30]:
df_test[feature_cols].to_parquet("test_featured.parquet", index=False)